https://www.bogotobogo.com/cplusplus/files/embed/OReilly_Programming_Embedded_Systems_Second_edition_ebook.pdf p161

https://stackoverflow.com/questions/13078394/whats-the-race-condition-in-these-two-interrupt-service-routines
https://www.stderr.nl/Blog/Hardware/Electronics/Arduino/Sleeping.html
stderr.nl(...) — Interrupts, sleeping and race conditions on ArduinoBlog post — Interrupts, sleeping and race conditions on Arduino https://www.reddit.com/r/embedded/comments/1p0gwdz/how_to_avoid_race_conditions_when_transmitting/

What about the ole 50 year beep bug?




https://users.cs.utah.edu/~regehr/papers/tv06.pdf

Interrupts have prioririties
Interrupts aren't interrupt by non-interrupt code

interrupts are atomic from non-interrupt perspective

spontaneous vs requested interrupt




a model of riscv interrupts?

spike seems like you can trgger interrupts from inside the code?

https://www.plantation-productions.com/Webster/www.artofasm.com/Windows/HTML/IOa4.html


mailbox recevies message, isr places in buffer. Main loop reads


python model?
tla
spin

using verilog model checker?


fdr
mcrl2


https://github.com/awslabs/shuttle

hypothesis / quickcheck style?

lean tla 


https://hypothesis.readthedocs.io/en/latest/stateful.html


benchmarks for simplifying (horrible) smt expressions
smt "decompilation" is something I encounter pretty often

Any tool that dumps smt?

Verifying at the c level does suck, yes.
But so does the assembly. Like I I could assume each instruction is atomic but that's not true.
deadlock empire


The atomic spec
The non atomic spec obviously has a flaw


Ah, wait. Since interrupt and mainloop are on the same core, we cna probably assume each instruction is atomic.

1. Is there a way to quickly model check this thing
2. 

ThreadSantizier + fuzzing?


In [15]:
%%file /tmp/race.tla
---------------- MODULE race ----------------

EXTENDS Naturals

VARIABLE gIndex

vars == <<gIndex>>

TypeOK == gIndex \in Nat

MainStep ==
    /\ gIndex # 0
    /\ gIndex' = gIndex - 1

IntStep == gIndex' = gIndex + 1

Init == gIndex = 0

Step ==
    \/ MainStep
    \/ IntStep

Spec == Init /\ [][Step]_vars

=============================================

Overwriting /tmp/race.tla


In [16]:
%%file /tmp/race.cfg

SPECIFICATION Spec

INVARIANT
    TypeOK



Overwriting /tmp/race.cfg


In [ ]:
import kdrag.solvers.tla as tla
import kdrag.smt as smt
#tla.check("/tmp/race.tla")

mod = tla.Module.of_file("/tmp/race")
mod.declare_var(smt.Int("gIndex"))
mod.infer_sorts
mod.action("Step").children()



[And(gIndex != 0, gIndex' == gIndex - 1), gIndex' == gIndex + 1]

In [1]:
from sail_graham import BinaryContext, FootprintRunner

runner = FootprintRunner.riscv64()
context = BinaryContext("program.elf", runner)
state = context.initial_state()

for trace in context.execute(context.entrypoint, state):
    solver = z3.Solver()
    solver.add(trace.path_condition)
    if solver.check() == z3.sat:
        print("next PC:", trace.state.pc)
        print("a0:", trace.state.regs["x10"])

CLEFileNotFoundError: Could not find file program.elf

In [ ]:
%%file /tmp/race2.tla
---------- MODULE race2 ------------------

(* --algorithm 

variables gIndex = 0;

proc Interrupt {
    gIndex += 1; \* Interrupt si not interrupted by mains thread. Atomic. _maybe_ Depends on memory model?
}

proc Main {
    while True {
        if gIndex {
            gIndex = gIndex - 1;
        }
    }
}

*)

Writing /tmp/race2.tla


In [6]:
import kdrag.solvers.tla as tla

tla.pluscal_translate("/tmp/race2.tla")

RuntimeError: tla2tools.jar failed with return code 255:
pcal.trans Version 1.12 of 01 July 2024

Unrecoverable error:
 -- Expected "begin" but found "gIndex"
    line 5, column 11.




In [44]:
%%file /tmp/race.c
int gIndex = 0;

__attribute__((interrupt)) void serialReceiveIsr(void)
{
    /* Store receive character in memory buffer. */
    gIndex++;
}

int main(void)
{
 while (1)
 {
    if (gIndex)
    {
        /* Process receive character in memory buffer. */
        gIndex--;
        //assert(gIndex >= 0);
    }
 }
}


Overwriting /tmp/race.c


In [45]:
! nix-shell -p pkgsCross.riscv32-embedded.buildPackages.gcc --run "riscv32-none-elf-gcc -fno-stack-protector -fno-pie -fno-builtin  -ffreestanding -nostdlib  -g -o /tmp/race /tmp/race.c && riscv32-none-elf-objdump -d -l -S /tmp/race"

/nix/store/qb0jzzkzm7bhjqs3a15nfb1n02jxsp4v-riscv32-none-elf-binutils-2.44/bin/riscv32-none-elf-ld: warning: cannot find entry symbol _start; defaulting to 000100b4

/tmp/race:     file format elf32-littleriscv


Disassembly of section .text:

000100b4 <serialReceiveIsr>:
serialReceiveIsr():
/tmp/race.c:4
int gIndex = 0;

__attribute__((interrupt)) void serialReceiveIsr(void)
{
   100b4:	1101                	addi	sp,sp,-32
   100b6:	ce06                	sw	ra,28(sp)
   100b8:	cc16                	sw	t0,24(sp)
   100ba:	ca22                	sw	s0,20(sp)
   100bc:	c83a                	sw	a4,16(sp)
   100be:	c63e                	sw	a5,12(sp)
   100c0:	1000                	addi	s0,sp,32
/tmp/race.c:6
    /* Store receive character in memory buffer. */
    gIndex++;
   100c2:	67c5                	lui	a5,0x11
   100c4:	1087a783          	lw	a5,264(a5) # 11108 <gIndex>
   100c8:	00178713          	addi	a4,a5,1
   100cc:	67c5                	lui	a5,0x11
   100ce:	10e7a423          	sw	a4,264(a

In [ ]:
import kdrag.contrib.pcode as pcode
import kdrag.smt as smt
ctx = pcode.BinaryContext("/tmp/race", "RISCV:LE:32:default")
ctx.loader.find_symbol("serialReceiveIsr").rebased_addr
ctx.loader.find_symbol("main").rebased_addr

def gIndex(memstate : pcode.MemState) -> smt.BitVecRef:
    return memstate.getvalue_ram(ctx.loader.find_symbol("gIndex").rebased_addr, 4)
    

mem0 = ctx.init_mem()
gIndex(mem0)



select32le(ram(state0), 69896)

# cbmc --isr

In [4]:
%%bash
goto-cc /tmp/race.c -o /tmp/main.gb
goto-instrument --isr serialReceiveIsr /tmp/main.gb /tmp/main_instrumented.gb
cbmc /tmp/main_instrumented.gb

Reading GOTO program from '/tmp/main.gb'
Function Pointer Removal
Virtual function removal
Cleaning inline assembler statements
Pointer Analysis
Instrumenting interrupt handler
Writing GOTO program to '/tmp/main_instrumented.gb'
CBMC version 6.8.0 (cbmc-6.8.0) 64-bit x86_64 linux
Reading GOTO program from file /tmp/main_instrumented.gb
Generating GOTO Program
Adding CPROVER library (x86_64)
Removal of function pointers and virtual functions
Generic Property Instrumentation
Starting Bounded Model Checking
Process was interrupted.


CalledProcessError: Command 'b'goto-cc /tmp/race.c -o /tmp/main.gb\ngoto-instrument --isr serialReceiveIsr /tmp/main.gb /tmp/main_instrumented.gb\ncbmc /tmp/main_instrumented.gb\n'' died with <Signals.SIGINT: 2>.

In [10]:
! gcc  -m64 -fno-stack-protector -fno-pie -fno-builtin -O0 -g -o /tmp/race /tmp/race.c

/tmp/race.c:4:1: error: interrupt service routine can only have a pointer argument and an optional integer argument
    4 | {
      | ^
/tmp/race.c: In function ‘serialReceiveIsr’:
/tmp/race.c:4:1: sorry, unimplemented: SSE instructions aren’t allowed in an interrupt service routine


# tsan pthread model
helgrind: thread
	valgrind --tool=helgrind --error-exitcode=66 $(BUILD)/race-thread

drd: thread
	valgrind --tool=drd --error-exitcode=66 $(BUILD)/race-thread

In [32]:
%%file /tmp/race1.c
#include <stdint.h>
#include <pthread.h>
#include <stdio.h>

// A global counter variable updated inside the ISR
volatile uint32_t gIndex = 0;
pthread_mutex_t gIndex_mutex = PTHREAD_MUTEX_INITIALIZER;


void mti_handler(void) {
    //pthread_mutex_lock(&gIndex_mutex);
    gIndex++;
    printf("gIndex: %d\n", gIndex);
    //pthread_mutex_unlock(&gIndex_mutex);

}

void* loop(void* arg) {
     while (1)
    {
        //pthread_mutex_lock(&gIndex_mutex);
        if (gIndex)
        {
            /* Process receive character in memory buffer. */

            gIndex--;
            printf("gIndex: %d\n", gIndex);
    }   
    //pthread_mutex_unlock(&gIndex_mutex);
 }
 return NULL;
}

void* interrupt_loop(void* arg) {
    while(1) {
        // random pause?
        mti_handler();
    }
    return NULL;
}


int main(){
    pthread_t main_t, interrupt_t;

    pthread_create(&main_t, NULL, loop, NULL);
    pthread_create(&interrupt_t, NULL, interrupt_loop, NULL);
    pthread_join(main_t, NULL); // won't happen
    return 0;
}

Overwriting /tmp/race1.c


In [34]:
! objdump -d /tmp/race1 


/tmp/race1:     file format elf64-x86-64


Disassembly of section .init:

0000000000001000 <_init>:
    1000:	f3 0f 1e fa          	endbr64
    1004:	48 83 ec 08          	sub    $0x8,%rsp
    1008:	48 8b 05 c9 2f 00 00 	mov    0x2fc9(%rip),%rax        # 3fd8 <__gmon_start__@Base>
    100f:	48 85 c0             	test   %rax,%rax
    1012:	74 02                	je     1016 <_init+0x16>
    1014:	ff d0                	call   *%rax
    1016:	48 83 c4 08          	add    $0x8,%rsp
    101a:	c3                   	ret

Disassembly of section .plt:

0000000000001020 <.plt>:
    1020:	ff 35 52 2f 00 00    	push   0x2f52(%rip)        # 3f78 <_GLOBAL_OFFSET_TABLE_+0x8>
    1026:	ff 25 54 2f 00 00    	jmp    *0x2f54(%rip)        # 3f80 <_GLOBAL_OFFSET_TABLE_+0x10>
    102c:	0f 1f 40 00          	nopl   0x0(%rax)
    1030:	f3 0f 1e fa          	endbr64
    1034:	68 00 00 00 00       	push   $0x0
    1039:	e9 e2 ff ff ff       	jmp    1020 <_init+0x20>
    103e:	66 90                	xchg   %ax,%a

In [ ]:
! gcc -fsanitize=thread -g -lpthread /tmp/race1.c -Wall -fanalyzer -o /tmp/race1 && setarch "$(uname -m)" -R /tmp/race1

# frama-c mthread

In [35]:
%%file /tmp/race.c
#include <stdint.h>

// A global counter variable updated inside the ISR
volatile uint32_t gIndex = 0;


void __attribute__((interrupt("machine"))) mti_handler(void) {
    gIndex++;
}

int main(void) {
     while (1)
    {
        if (gIndex)
        {
            /* Process receive character in memory buffer. */
            gIndex--;
    }   
 }

}

Overwriting /tmp/race.c


In [37]:
! frama-c /tmp/race.c -mthread -mt-interrupt-handlers=mti_handler -mt-write-races

[mt] Preparing sources for Mthread with builtins only
[kernel] Parsing FRAMAC_SHARE/mt/mthread.c (with preprocessing)
[kernel] Parsing /tmp/race.c (with preprocessing)
[kernel:unknown-attribute] /tmp/race.c:7: Warning: 
  Ignoring unknown attribute: interrupt
[mt] Warning: Mthread is an experimental plugin and is still in development.
[mt] ******* Starting mthread
[mt] *** Computing value analysis for main thread
[eva] Found concurrency builtins: enabling mthread domain
[eva:experimental] Warning: The mthread domain is experimental.
[eva] Analyzing a complete application starting at main
[eva:initial-state] Values of globals at initialization
  __fc_mthread_threads_running ∈ {0}
  __fc_mthread_threads[0..31] ∈ {0}
  __fc_mthread_mutexes[0..31] ∈ {0}
  __fc_mthread_queues[0..31] ∈ {0}
  gIndex ∈ [--..--]
[mt] New thread: <main>, fun main
[mt] New thread: <interrupt_handler mti_handler>, fun mti_handler
[mt] New context for <main>, fun main
[mt] New context for <interrupt_handler mti_han

In [6]:
! opam exec --switch=5.3.0 -- frama-c /tmp/race.c -mthread -mt-h

Plug-in name: mthread
Plug-in shortname: mt
Description: Experimental tools for multi-threaded programs

Most options of the form '-mt-option-name' and without any parameter
have an opposite with the name '-mt-no-option-name'.

Most options of the form '-option-name' and without any parameter
have an opposite with the name '-no-option-name'.

Options taking a string as argument should preferably be written
-option-name="argument".

***** LIST OF AVAILABLE OPTIONS:

-mt-share <dir>     set the plug-in share directory to <dir> (may be used if
                    the plug-in is not installed at the same place as
                    Frama-C)
-mthread            enable analysis of multi-threaded programs through the
                    Mthread plugin (opposite option is -no-mthread)

*** ANALYSIS

-mt-interrupt-handlers <functions>  Specify functions that will be treated as
                    handlers for interrupts. (preferably use
                    -mt-interrupt-handlers="functions")
-

In [5]:
! opam exec --switch=5.3.0 -- frama-c /tmp/race.c -mthread -mt-interrupt-handlers=mti_handler -mt-write-races

[kernel] Parsing FRAMAC_SHARE/mt/mthread.h (with preprocessing)
[kernel] Parsing /tmp/race.c (with preprocessing)
[kernel:unknown-attribute] /tmp/race.c:7: Warning: 
  Ignoring unknown attribute: interrupt
[mt] Warning: Mthread is an experimental plugin and is still in development.
[mt] ******* Starting mthread
[mt] *** Computing value analysis for main thread
[eva] Analyzing a complete application starting at main
[eva:initial-state] Values of globals at initialization
  gIndex ∈ [--..--]
[mt] New thread: <main>, fun main
[mt] New thread: <interrupt_handler mti_handler>, fun mti_handler
[mt] New context for <main>, fun main
[mt] New context for <interrupt_handler mti_handler>, fun mti_handler
[eva:summary] ====== ANALYSIS SUMMARY ======
  ----------------------------------------------------------------------------
  1 function analyzed (out of 2): 50% coverage.
  In this function, 3 statements reached (out of 4): 75% coverage.
  --------------------------------------------------------

In [40]:
! nix-shell -p pkgsCross.riscv32-embedded.buildPackages.gcc --run "riscv32-none-elf-gcc -fno-stack-protector -fno-pie -fno-builtin  -ffreestanding -nostdlib  -g -o /tmp/race /tmp/race.c && riscv32-none-elf-objdump -d -l -S /tmp/race"

/tmp/race.c: In function ‘main’:
/tmp/race.c:17:9: error: implicit declaration of function ‘assert’ []8;;https://gcc.gnu.org/onlinedocs/gcc-14.3.0/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   17 |         assert(gIndex >= 0);
      |         ^~~~~~
/tmp/race.c:1:1: note: ‘assert’ is defined in header ‘<assert.h>’; this is probably fixable by adding ‘#include <assert.h>’
  +++ |+#include <assert.h>
    1 | int gIndex = 0;


In [3]:
! nix-shell -p pkgsCross.riscv32-embedded.buildPackages.gcc --run "riscv32-none-elf-gcc -fno-stack-protector -fno-pie -fno-builtin  -ffreestanding -nostdlib  -O0 -g -o /tmp/race /tmp/race.c && riscv32-none-elf-objdump -d -l -S /tmp/race"

/nix/store/qb0jzzkzm7bhjqs3a15nfb1n02jxsp4v-riscv32-none-elf-binutils-2.44/bin/riscv32-none-elf-ld: warning: cannot find entry symbol _start; defaulting to 000100b4

/tmp/race:     file format elf32-littleriscv


Disassembly of section .text:

000100b4 <mti_handler>:
mti_handler():
/tmp/race.c:7

// A global counter variable updated inside the ISR
volatile uint32_t gIndex = 0;


void __attribute__((interrupt("machine"))) mti_handler(void) {
   100b4:	1101                	addi	sp,sp,-32
   100b6:	ce06                	sw	ra,28(sp)
   100b8:	cc16                	sw	t0,24(sp)
   100ba:	ca22                	sw	s0,20(sp)
   100bc:	c83a                	sw	a4,16(sp)
   100be:	c63e                	sw	a5,12(sp)
   100c0:	1000                	addi	s0,sp,32
/tmp/race.c:8
    gIndex++;
   100c2:	67c5                	lui	a5,0x11
   100c4:	1087a783          	lw	a5,264(a5) # 11108 <gIndex>
   100c8:	00178713          	addi	a4,a5,1
   100cc:	67c5                	lui	a5,0x11
   100ce:	10e7a423          

In [27]:
! nix-shell -p pkgsCross.riscv32-embedded.buildPackages.gcc --run "riscv32-none-elf-gcc -fno-stack-protector -fno-pie -fno-builtin  -ffreestanding -nostdlib  -O0 -g -o /tmp/race /tmp/race.c && riscv32-none-elf-objdump -d -l /tmp/race"

/nix/store/qb0jzzkzm7bhjqs3a15nfb1n02jxsp4v-riscv32-none-elf-binutils-2.44/bin/riscv32-none-elf-ld: warning: cannot find entry symbol _start; defaulting to 000100b4

/tmp/race:     file format elf32-littleriscv


Disassembly of section .text:

000100b4 <mti_handler>:
mti_handler():
/tmp/race.c:7
   100b4:	1101                	addi	sp,sp,-32
   100b6:	ce06                	sw	ra,28(sp)
   100b8:	cc16                	sw	t0,24(sp)
   100ba:	ca22                	sw	s0,20(sp)
   100bc:	c83a                	sw	a4,16(sp)
   100be:	c63e                	sw	a5,12(sp)
   100c0:	1000                	addi	s0,sp,32
/tmp/race.c:8
   100c2:	67c5                	lui	a5,0x11
   100c4:	1087a783          	lw	a5,264(a5) # 11108 <gIndex>
   100c8:	00178713          	addi	a4,a5,1
   100cc:	67c5                	lui	a5,0x11
   100ce:	10e7a423          	sw	a4,264(a5) # 11108 <gIndex>
/tmp/race.c:9
   100d2:	0001                	nop
   100d4:	40f2                	lw	ra,28(sp)
   100d6:	42e2                	lw	t0,

# tla

# verilog

In [ ]:
%%file /tmp/mailbox.v

module mailbox(
    input send
    //input interrupt,
    input clk,
)

reg full;
reg [7:0] buffer;
reg [3:0] index;
reg ack;
reg int_enable;

/* SENDER
nondeterminstically attempt to send another chunk through the mailbox
*/
always @(posedge send) begin
    if !full begin
        full <= 1;
    end
end

/* 
ISR
triggered by mailbox being full
But also enable?
*/
always @(posedge full and int_enable or posedge int_enable and full) begin
        full <= 0;
        index <= index + 1;
        
        if PC == WFI begin // trigger leaving wfi state
            PC <= check;
        end
end

PC = {CHECK, WFI, };
// RECEIVER
always @(posedge clk) begin
    match PC
        switch CHECK:
            PC <= WFI;
        switch WFI;
end


endmodule



# ghidra


In [31]:
%%file /tmp/int.s
start:
    csrr t1, mstatus 
    csrs mie, t1 
    wfi
    mret
    add t1,t2,t3


Overwriting /tmp/int.s


In [32]:
! riscv64-unknown-elf-as -o /tmp/int.o /tmp/int.s && riscv64-unknown-elf-objdump -d /tmp/int.o


/tmp/int.o:     file format elf64-littleriscv


Disassembly of section .text:

0000000000000000 <start>:
   0:	30002373          	csrr	t1,mstatus
   4:	30432073          	csrs	mie,t1
   8:	10500073          	wfi
   c:	30200073          	mret
  10:	01c38333          	add	t1,t2,t3


In [41]:
from pypcode import *
ctx = Context("RISCV:LE:64:RV64IC")
machine_code = bytes.fromhex("30002373")
machine_code = bytes.fromhex("30200073 ")
machine_code = bytes.fromhex("01c38333")
machine_code = bytes.fromhex("3383c301")
machine_code = bytes.fromhex("73230030")
machine_code = bytearray.fromhex("30432073")
machine_code.reverse()
machine_code = bytes(machine_code)
disasm = ctx.disassemble(bytes(machine_code))
for insn in disasm.instructions:
    print(insn)
translation = ctx.translate(machine_code)
for op in translation.ops:
    print(op)
    #print(op.opcode, op.output.space.name if op.output else None, op.inputs)

0x0/4: csrrs zero,mie,t1
IMARK ram[0:4]
unique[12880:8] = t1
zero = mie
unique[12a00:1] = 0x6 == 0x0
if (unique[12a00:1]) goto ram[4:8]
mie = mie | unique[12880:8]


In [ ]:
bit int_enable;
bit mailbox_full;


active proc sender(){

}

active proc isr(){

}

active proc mainloop(){
    int_enable = 0;

    int_enable = 1;
    wfi
}

In [ ]:


while True:
    i = random.randint()
    match i:
        case 0:
            messager.step()
        case 1:
            isr.step()
        case 2:
            mainloop.step()



In [ ]:
mstatus = 0
mie = 

def can_fire
def 



# pcode + high level

TLA spec of interrupt behavior. Refine onto assembly state


```tla

VARIABLE mie

MieEnable == mie' = True

```

```python
#def high_low(x : App) -> :
#    match x:
#        case App("mie", []):
#            return 

def high_low(x : Memstate) -> dict[str, smt.ExprRef]:
    return {
        "mie" : memstate.readregister("MIE") == 0,
        "
    }

def bisim(mod,  actions : list[str], decls, high_low):



def bisim_c():


```


```
def emit_tla

```

